In [2]:
# import geopandas as gpd
# import pandas as pd
# import numpy as np

import sys
sys.path.append("..")
# %load_ext autoreload
# %autoreload 2
# %reload_ext autoreload

# from src.data import *

# from src.data.make_dataset import zoning_layer
# print(zoning_layer.head())




from src.features.nested_functions import *
from src.data.make_dataset import *


#from src.data.make_dataset import *
#from src.features.nested_functions import *
#from src.features.indicator_functions import *
#from src.features.criteria_functions import *
#from src.data.weights import *


#print(myfunc(test_par_df.loc['res_type']))
#test_par_df['lu_label'] = test_par_df.apply(lambda row: lu_dict_test(res_type= row['res_type'],
#                                                                     luc= row['luc']), axis=1)


#test_par_df['use_lamda'] = test_par_df.apply(lamda row: row+  )

#print(test_par_df['lu_label'])


c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP.
  return ogr_read(
c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  ret

In [3]:
town_name = 'Quincy'
muni_parcels = get_landuse_data(town_name)

#get muni boundary + buffer
muni_gdf = ma_towns.loc[ma_towns['TOWN'].str.casefold() == town_name.casefold()].to_crs(mass_mainland_crs)
muni_gdf_w_buffer = buffer_gdf(muni_gdf, 800)

In [5]:
res_zoning = get_zoning_data(town_name)

In [ ]:
parcels_gdf = muni_parcels
zoning_gdf = res_zoning

#def zoning_merge(zoning_gdf, parcels_gdf):
par_zon_join = parcels_gdf.overlay(zoning_gdf, how = "intersection", keep_geom_type = True)

test = len(par_zon_join['LOC_ID']) >  len(set(par_zon_join['LOC_ID']))
print(test)
# determine the proportion of the total parcel area
par_zon_join['zone_area'] = par_zon_join['geometry'].area
par_zon_join['zone_share'] = par_zon_join.apply(lambda row: row['zone_area']*10.7639/row['LOT_SIZE_GIS'], axis=1) #its a row

In [52]:
# ID the index of the row with the largest share of a parcel in a zone for each unique LOCID
idx = par_zon_join.fillna(999999).groupby('LOC_ID')['zone_share'].idxmax().reset_index()

# subset the original spatial join to the rows where the share is the largest for each unique LOC ID--should be one row for each LOCID again
clean_join = par_zon_join.loc[idx['zone_share']]
# cut that table to just the LOC ID and the ZO Code, confirm pd
par_zon_xwalk = clean_join[['LOC_ID','ZO_CODE']]
print(par_zon_xwalk.info())
# take the zoning input and get rid of the geometry so we can do non-spatial joins
zoning_table = pd.DataFrame(zoning_gdf.drop(columns= 'geometry')) #confirm geom column name
# join the zone code onto the parcels, double check the join
parcels_zone_rec = pd.merge(parcels_gdf, par_zon_xwalk, left_on= 'LOC_ID', right_on= "LOC_ID", how = "left")
print(len(parcels_zone_rec['LOC_ID']) == len(clean_join))
print(parcels_zone_rec.info())
# # join the rest of the zoning table back to the parcels
par_zon_join_fixed = pd.merge(parcels_zone_rec, zoning_table, left_on= 'ZO_CODE', right_on= 'ZO_CODE', how = "inner")
print("Zones assigned to Parcels by Largest Share")
# # return par_zon_join_fixed

# else : 
# print("Parcels Sucessfully Spatial Joined")
# #return par_zon_join

# #create the zoning layer






<class 'pandas.core.frame.DataFrame'>
Index: 16325 entries, 17890 to 17775
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   LOC_ID   16325 non-null  object
 1   ZO_CODE  16325 non-null  object
dtypes: object(2)
memory usage: 382.6+ KB
None
False
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 16329 entries, 0 to 16328
Data columns (total 62 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   LOC_ID              16329 non-null  object  
 1   geometry            16329 non-null  geometry
 2   Unnamed: 0          16329 non-null  int64   
 3   TOWN_ID             16329 non-null  int64   
 4   PROP_ID             16329 non-null  object  
 5   BLDG_VAL            16329 non-null  int64   
 6   LAND_VAL            16329 non-null  int64   
 7   OTHER_VAL           16329 non-null  int64   
 8   TOTAL_VAL           16329 non-null  int64   
 9   FY                  

In [ ]:

clean_join
len(idx) == len(clean_join)

True

In [ ]:
muni_parcels

CASTING POLYGON TO MULTIPOLYGON

In [24]:
from shapely import MultiPolygon
par_zon_join["geometry"] = [MultiPolygon([feature]) if isinstance(feature, Polygon) \
    else feature for feature in par_zon_join["geometry"]]
#q_export = MultiPolygon(par_zon_join['geometry'])
hmm = par_zon_join.geom_type
par_zon_join.to_file(zoning_gdb, layer = "quincy_par_zo_test", driver = 'OpenFileGDB')

In [53]:
from shapely import MultiPolygon
par_zon_join_fixed["geometry"] = [MultiPolygon([feature]) if isinstance(feature, Polygon) \
    else feature for feature in par_zon_join_fixed["geometry"]]
#q_export = MultiPolygon(par_zon_join['geometry'])
par_zon_join_fixed.to_file(zoning_gdb, layer = "quincy_par_zo_test_fixed", driver = 'OpenFileGDB')

In [3]:
import geopandas as gpd
import os
import pandas as pd
datasets_dir = r'K:\DataServices\Datasets'
projects_dir = 'K:\\DataServices\\Projects\\Current_Projects'


zoning_gdb = os.path.join(datasets_dir, r"Zoning and Land Use\Town_Zoning\ZoningData\zoning_gdb.gdb")
zoning_layer = gpd.read_file(zoning_gdb, layer = 'mmc_zoning')  

zoning_layer.head()

,oid_,zo_code,muni_id,muni,zo_abbr,zo_name,shape_leng,Shape_Length,Shape_Area,geometry
0,23.0,163CBD,163.0,Lynn,CBD,Central Business District,20739.464212,20739.464212,5.162023e+05,"MULTIPOLYGON (((245849.080 912481.350, 245881...."
1,28.0,207BU1,207.0,Newton,BU1,Business 1,28428.723494,28428.723494,5.058131e+05,"MULTIPOLYGON (((226792.422 896888.775, 226793...."
2,30.0,207LM,207.0,Newton,LM,Light manufacturing,6656.465850,6656.465850,4.515634e+05,"MULTIPOLYGON (((224989.655 893478.826, 224991...."
3,31.0,207SR3,207.0,Newton,SR3,Single Residence 3,245884.695433,245884.695433,6.283705e+06,"MULTIPOLYGON (((223042.124 901036.730, 223047...."
4,69.0,40GB,40.0,Braintree,GB,General Business,27594.178038,27594.178038,5.093180e+05,"MULTIPOLYGON (((242429.738 882230.167, 242429...."


In [ ]:
import pyogrio

pyogrio.list_drivers() 


In [ ]:
from src.features.zoning_nonconformity_scripts import *
from src.data.make_dataset import *

test_output = run_zoning_nonconformity('Newton')

print(test_output.head())

In [56]:
project_dir = r'C:\Users\ziacovino\Desktop\temp_muni'
def get_most_updated_state_assessors_data(muni):

        '''
        pulls parcel data from the state's assessors website (with an exception for Boston who hosts separately)
        '''

        
        if muni == 'Boston':
            #get shapefile from boston's open data portal and download
            #change in make_dataset.py

            parcel_layer = boston_parcels.copy()

            parcel_layer = parcel_layer.loc[parcel_layer['POLY_TYPE']=='FEE']
            return(parcel_layer)
        

        else:

            #get shapefile from a massgis link
            shapefile_excel = 'https://www.mass.gov/doc/massgis-parcel-data-download-links-table/download'
            shapefile_lookup = pd.read_excel(shapefile_excel)

            town_shp_lookup_link = shapefile_lookup.loc[shapefile_lookup['Town Name'] == muni.upper()]

            #extract into a temporary folder for use
            
            path = os.path.join(project_dir, 'Data', muni) #make a subdirectory in ortho folder w town name
            os.makedirs(path, exist_ok=True) 

            for url in town_shp_lookup_link['Shapefile Download URL'].tolist():
                with urlopen(url) as zipresp:
                    with zipfile.ZipFile(BytesIO(zipresp.read())) as zfile:
                        zfile.extractall(path)


            layer = get_file(path, 'TaxPar', '.shp')
            parcel_layer = gpd.read_file(layer)

            parcel_layer = parcel_layer.loc[parcel_layer['POLY_TYPE']=='FEE']
            return(parcel_layer)

muni_state_parcels = get_most_updated_state_assessors_data(town_name)

mapc pdb: 21,186
mass gis : 16,425
joined records : 16,329
spatial join recoreds: 16,325

In [ ]:
muni_state_parcels

In [9]:
zoning_project_dir = r'C:\Users\ziacovino\OneDrive - Metropolitan Area Planning Council\Metro Mayors Housing Task Force\Phase 2 Scope of Work\Rightsizing Zoning Project\Data'
zoning = zoning_layer[zoning_layer['muni'] == town_name]
zoning = zoning.to_crs(mass_mainland_crs)
#zo_code = 'Zoning'
# regulation table (exported 4/10)
reg_table_fp = os.path.join(zoning_project_dir, "zoning-atlas-mmc.csv")
#original Newton table
#reg_table_fp = os.path.join(zoning_project_dir, "zoning-regs-by_right.csv")
reg_table = pd.read_csv(reg_table_fp)
zoning_reg_table = pd.merge(zoning, reg_table, left_on= 'zo_code', right_on= "ZO_CODE", how = "inner")

In [5]:
town_name = "Quincy"
muni_parcels = get_landuse_data(town_name)

res_zoning = get_zoning_data(town_name)

muni_w_zoning = zoning_merge(zoning_gdf= res_zoning,
                                 parcels_gdf= muni_parcels)

Split Zoned Parcels Detected
Clean Join successful?
True
Zone Code succesfully joined?
False
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 16329 entries, 0 to 16328
Data columns (total 64 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   LOC_ID              16329 non-null  object  
 1   geometry            16329 non-null  geometry
 2   Unnamed: 0          16329 non-null  int64   
 3   TOWN_ID             16329 non-null  int64   
 4   PROP_ID             16329 non-null  object  
 5   BLDG_VAL            16329 non-null  int64   
 6   LAND_VAL            16329 non-null  int64   
 7   OTHER_VAL           16329 non-null  int64   
 8   TOTAL_VAL           16329 non-null  int64   
 9   FY                  16329 non-null  object  
 10  LOT_SIZE            16329 non-null  float64 
 11  LS_DATE             16329 non-null  object  
 12  LS_PRICE_S          16329 non-null  int64   
 13  LS_PRICE_L          16329 non-null 

In [6]:
lidar_bld = gpd.read_file(r'K:\DataServices\Projects\Current_Projects\Climate_Change\MVP_MMC_CoolRoofs_MVP\Data\Analysis_Data\Data_Cool_Roofs\2_Output\MMC_Cool_Roofs.shp')

In [26]:
ldr_sub = lidar_bld[['LOC_ID','CITY', 'flat_roof', 'geometry']]

ldr_town = ldr_sub[ldr_sub['CITY'] == town_name]

ldr_town = ldr_town.rename(columns= {'LOC_ID':'LOC_ID_bld'})


In [16]:
intersection_layer

<Projected CRS: EPSG:26986>
Name: NAD83 / Massachusetts Mainland
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - Massachusetts onshore - counties of Barnstable; Berkshire; Bristol; Essex; Franklin; Hampden; Hampshire; Middlesex; Norfolk; Plymouth; Suffolk; Worcester.
- bounds: (-73.5, 41.46, -69.86, 42.89)
Coordinate Operation:
- name: SPCS83 Massachusetts Mainland zone (meter)
- method: Lambert Conic Conformal (2SP)
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [44]:
intersection_layer = muni_parcels.overlay(ldr_town, how='intersection', keep_geom_type=True)
        
#get area of overlap for the area of intersection
intersection_layer['lot_cov' + '_sqm'] = intersection_layer['geometry'].area  
intersection_layer = intersection_layer.groupby(by=id_field).agg({('lot_cov' + '_sqm'):'sum'}).reset_index()

In [45]:
from src.features.indicator_functions import *
muni_parcels = calculate_overlap(layer_1= muni_parcels,
                                                        layer_2 = ldr_town,
                                                        how = "percent",
                                                    new_field_name= 'par_cov')

In [43]:
from shapely import MultiPolygon
from shapely import Point
intersection_layer["geometry"] = [MultiPolygon([feature]) if isinstance(feature, Polygon)
    else feature for feature in intersection_layer["geometry"]]
#q_export = MultiPolygon(par_zon_join['geometry'])
intersection_layer.to_file(zoning_gdb, layer = "quincy_bld_ovp_test", driver = 'OpenFileGDB')

c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field Unnamed: 0 of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed__0'
  ogr_write(
c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field TOWN_ID of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
c:\Users\ZIacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field BLDG_VAL of type Integer64 will be written as a Float64. To get Integer64, use layer creation 

In [50]:
muni_parcels['DUA'] = muni_parcels[units]/(muni_parcels['LOT_SIZE_GIS']/43560)